# GenImage Full-Resolution Detector — Phase 2

Trains a ResNet50V2 detector on several GenImage generators (224x224) and tests it on a
**held-out generator it never saw**.

**Credentials:** Kaggle username/key are read from Colab Secrets (sidebar 🔑 icon — add
`KAGGLE_USERNAME` and `KAGGLE_KEY`) or prompted at runtime — nothing sensitive is stored in this file.
**Paths:** everything is written under `OUTPUT_DIR` (default local). Point it at a Drive folder
if you want persistence. Set Runtime -> GPU (T4) before running.


## 0. Setup

In [1]:
import tensorflow as tf
from tensorflow.keras import layers
import os
print('GPU:', tf.config.list_physical_devices('GPU'))

# Optional: if you set OUTPUT_DIR to a Drive path below, mount Drive first:
# from google.colab import drive; drive.mount('/content/drive')


GPU: []


## 1. Configuration

In [ ]:
# ---- Kaggle credentials (no secrets committed) ----
try:
    from google.colab import userdata               # Colab Secrets (recommended)
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
except Exception:
    import getpass                                   # fallback: prompt at runtime
    os.environ.setdefault("KAGGLE_USERNAME", input("Kaggle username: "))
    os.environ.setdefault("KAGGLE_KEY", getpass.getpass("Kaggle key: "))

# ---- Where to cache data + save the model ----
# Local by default. For persistence, set e.g. "/content/drive/MyDrive/genimage" (mount Drive first).
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/content/genimage_out")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Experiment settings ----
TRAIN_GENERATORS  = ["stable_diffusion_v_1_4", "glide", "BigGAN"]
HELDOUT_GENERATOR = "wukong"
IMG, BATCH, FINE_TUNE = (224, 224), 32, False

DATA_DIR  = "/content/genimage_val"
BASE      = f"{DATA_DIR}/GenImage"
CACHE_ZIP = f"{OUTPUT_DIR}/genimage_validation.zip"
MODEL_OUT = f"{OUTPUT_DIR}/genimage_model.keras"


## 2. Get the data (download once, cache to OUTPUT_DIR)

In [ ]:
if os.path.isdir(BASE):
    print("Data already extracted at", BASE)
elif os.path.exists(CACHE_ZIP):
    print("Restoring from cache...")
    get_ipython().system(f"unzip -q '{CACHE_ZIP}' -d {DATA_DIR}")
else:
    print("First-time download from Kaggle...")
    get_ipython().system("pip install -q kaggle")
    get_ipython().system("kaggle datasets download -d vtphatt2/genimage-validation -p /content")
    get_ipython().system(f"unzip -q -o /content/genimage-validation.zip -d {DATA_DIR}")
    get_ipython().system(f"cp /content/genimage-validation.zip '{CACHE_ZIP}'")
get_ipython().system(f"ls {BASE}")


## 3. Build datasets (internal train / val / held-out split)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE   # class order alphabetical -> ai=0, nature=1 (fake=0, real=1)

def gen_split(gen, subset, split=0.2):
    return tf.keras.utils.image_dataset_from_directory(
        f"{BASE}/{gen}/val", label_mode="binary", image_size=IMG, batch_size=BATCH,
        validation_split=split, subset=subset, seed=42)

def concat(parts):
    d = parts[0]
    for p in parts[1:]:
        d = d.concatenate(p)
    return d

# TRAIN: randomly interleave generators (good mixing, low memory — no big shuffle buffer / no cache at 224px)
train_ds = tf.data.Dataset.sample_from_datasets(
    [gen_split(g, "training") for g in TRAIN_GENERATORS], stop_on_empty_dataset=False).prefetch(AUTOTUNE)
val_ds   = concat([gen_split(g, "validation") for g in TRAIN_GENERATORS]).prefetch(AUTOTUNE)
heldout_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/{HELDOUT_GENERATOR}/val", label_mode="binary", image_size=IMG,
    batch_size=BATCH, shuffle=False).prefetch(AUTOTUNE)
print("datasets ready")


## 4. Model — ResNet50V2 transfer learning

In [ ]:
base = tf.keras.applications.ResNet50V2(include_top=False, weights="imagenet", input_shape=(*IMG, 3))
base.trainable = FINE_TUNE

model = tf.keras.Sequential([
    layers.Input(shape=(*IMG, 3)),
    layers.Rescaling(1./127.5, offset=-1),   # ResNetV2 preprocessing
    layers.RandomFlip("horizontal"),
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5 if FINE_TUNE else 1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


## 5. Train

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
cbs = [ModelCheckpoint(MODEL_OUT, monitor="val_loss", save_best_only=True, verbose=1),
       EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
       ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)]
history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cbs)
model.save(MODEL_OUT); print("saved to", MODEL_OUT)


## 6. Generalization test

In [ ]:
_, indist = model.evaluate(val_ds, verbose=0)
_, held   = model.evaluate(heldout_ds, verbose=0)
print(f"In-distribution: {indist:.4f}")
print(f"Held-out ({HELDOUT_GENERATOR}): {held:.4f}")
print(f"Generalization gap: {indist - held:.4f}")
